Test the two methods of calculating depth.

In [1]:
import xarray as xr
import multiprocessing as mp
import pandas as pd
import numpy as np
import time

from depth_by_geometry import solve_depth_geom
from depth_from_flow_cc import solve_depth

In [2]:
chrtout_path = "test/nwm.20260219_analysis_assim_nwm.t15z.analysis_assim.channel_rt.tm02.conus.nc"
routelink_path = "test/RouteLink_CONUS.nc"

In [3]:
chrtout = xr.open_dataset(chrtout_path)
routelink = xr.open_dataset(routelink_path)

In [4]:
chrtout

<xarray.Dataset> Size: 155MB
Dimensions:         (feature_id: 2776734, time: 1, reference_time: 1)
Coordinates:
  * feature_id      (feature_id) int64 22MB 101 179 ... 1180001803 1180001804
  * time            (time) datetime64[ns] 8B 2026-02-19T13:00:00
  * reference_time  (reference_time) datetime64[ns] 8B 2026-02-19T12:00:00
Data variables:
    crs             |S1 1B ...
    streamflow      (feature_id) float64 22MB ...
    nudge           (feature_id) float64 22MB ...
    velocity        (feature_id) float64 22MB ...
    qSfcLatRunoff   (feature_id) float64 22MB ...
    qBucket         (feature_id) float64 22MB ...
    qBtmVertRunoff  (feature_id) float64 22MB ...
Attributes: (12/19)
    TITLE:                      OUTPUT FROM NWM v3.0
    featureType:                timeSeries
    proj4:                      +proj=lcc +units=m +a=6370000.0 +b=6370000.0 ...
    model_initialization_time:  2026-02-19_12:00:00
    station_dimension:          feature_id
    model_output_valid_time:    2026-02-19_13:00:00
    ...                         ...
    model_configuration:        analysis_and_assimilation
    dev_OVRTSWCRT:              1
    dev_NOAH_TIMESTEP:          3600
    dev_channel_only:           0
    dev_channelBucket_only:     0
    dev:                        dev_ prefix indicates development/internal me...

In [5]:
routelink

<xarray.Dataset> Size: 269MB
Dimensions:            (feature_id: 2776734)
Coordinates:
    lon                (feature_id) float32 11MB ...
    lat                (feature_id) float32 11MB ...
Dimensions without coordinates: feature_id
Data variables: (12/21)
    link               (feature_id) int32 11MB ...
    from               (feature_id) int32 11MB ...
    to                 (feature_id) int32 11MB ...
    alt                (feature_id) float32 11MB ...
    order              (feature_id) int32 11MB ...
    Qi                 (feature_id) float32 11MB ...
    ...                 ...
    gages              (feature_id) |S15 42MB ...
    Kchan              (feature_id) int16 6MB ...
    ascendingIndex     (feature_id) int32 11MB ...
    nCC                (feature_id) float32 11MB ...
    TopWdthCC          (feature_id) float32 11MB ...
    TopWdth            (feature_id) float32 11MB ...
Attributes:
    Convention:        CF-1.6
    featureType:       timeSeries
    history:           Created Thu Sep  9 18:11:34 2021
    processing_notes:  This file was produced Thu Sep  9 16:16:38 2021 by Kev...

In [6]:
gages_bytes = routelink["gages"].values
gages_temp = [gage.decode("utf-8") for gage in gages_bytes]
gages_temp = ["None" if g == '               ' else g.strip() for g in gages_temp]
has_gage = [True if g != "None" else False for g in gages_temp]
# gages = gages[has_gage]
gages = np.array(gages_temp)[has_gage]

In [7]:
feature_ids = routelink["link"].values[has_gage]

The geometry-only method (assumes correct velocity from CHRTOUT)

In [8]:
# test depth_by_geometry
streamflow = chrtout["streamflow"].sel(feature_id=feature_ids)
velocity = chrtout["velocity"].sel(feature_id=feature_ids)
tw = routelink["TopWdth"].values[has_gage]
bw = routelink["BtmWdth"].values[has_gage]
cs = routelink["ChSlp"].values[has_gage]

start_time = time.perf_counter()
depths_geom = solve_depth_geom(streamflow, velocity, tw, bw, cs)
end_time = time.perf_counter()
print(f"Depths by geometry calculated in {end_time - start_time:.4f} seconds")

Depths by geometry calculated in 2.0467 seconds


In [9]:
depths_geom_df = pd.DataFrame({"feature_id": feature_ids, "gage": gages, "depth_geom": depths_geom})
depths_geom_df.to_parquet("test/depths_by_geometry.parquet", index=False)
depths_geom_df.head()

,feature_id,gage,depth_geom
0,7086109,05106000,0.127322
1,7040481,05078520,0.026219
2,7053819,05078470,0.079413
3,7111205,05125039,0.030890
4,7110249,05124982,0.000000


The iterative Manning's equation method (velocity not used)

In [10]:
# test depth_from_flow_cc
n = routelink["n"].values[has_gage]
ncc = routelink["nCC"].values[has_gage]
s0 = routelink["So"].values[has_gage]

start_time = time.perf_counter()
depths_manning = solve_depth(streamflow, bw, tw, cs, n, ncc, s0)
end_time = time.perf_counter()
print(f"Depths by iterating Manning's equation calculated in {end_time - start_time:.4f} seconds")

Depths by iterating Manning's equation calculated in 15.2818 seconds


In [11]:
depths_mann_df = pd.DataFrame({"feature_id": feature_ids, "gage": gages,
"depth_mann": depths_manning})
depths_mann_df.to_parquet("test/depths_by_mann.parquet", index=False)
depths_mann_df.head()

,feature_id,gage,depth_mann
0,7086109,05106000,0.130416
1,7040481,05078520,0.029665
2,7053819,05078470,0.076068
3,7111205,05125039,0.026856
4,7110249,05124982,0.000000


In [12]:
depths_geom_ft = depths_geom * 3.28084
depths_mann_ft = depths_manning * 3.28084

In [13]:
diff = abs(depths_geom_ft - depths_mann_ft)

In [14]:
max(diff)
min(diff)
mean_diff = np.mean(diff)
print(f"Max difference: {max(diff):.4f} ft")
print(f"Min difference: {min(diff):.4f} ft")
print(f"Mean difference: {mean_diff:.4f} ft")

Max difference: 1281.7415 ft
Min difference: 0.0000 ft
Mean difference: 1.2574 ft


In [15]:
most_diff = np.sort(diff)[::-1][:30]
print("Top 10 differences:", most_diff)

Top 10 differences: [1281.74153459  664.05552209  295.50527543   91.51965594   77.82759608
   60.45589929   53.31563155   52.87349373   51.67780552   51.56838415
   43.7582219    29.74893429   29.4803446    29.10876373   25.59978766
   24.67761738   24.64648888   23.30401376   22.89614949   22.70888545
   22.6231669    20.73242271   20.7077776    20.51548822   19.98113509
   19.95847597   19.66093051   18.20983816   16.9128975    16.82983638]


In [16]:
for i in range(30):
    idx = np.where(diff == most_diff[i])[0][0]
    print(f"Feature ID: {feature_ids[idx]}, Gage: {gages[idx]}, Depth by Geometry: {depths_geom_ft[idx]:.4f} ft, Depth by Manning's: {depths_mann_ft[idx]:.4f} ft, Difference: {diff[idx]:.4f} ft")

Feature ID: 19406818, Gage: 07381482, Depth by Geometry: 126.2247 ft, Depth by Manning's: 1407.9662 ft, Difference: 1281.7415 ft
Feature ID: 24092756, Gage: 11504115, Depth by Geometry: 30.0390 ft, Depth by Manning's: 694.0945 ft, Difference: 664.0555 ft
Feature ID: 22720801, Gage: 02196485, Depth by Geometry: 39.1844 ft, Depth by Manning's: 334.6897 ft, Difference: 295.5053 ft
Feature ID: 13196034, Gage: 04159130, Depth by Geometry: 0.0000 ft, Depth by Manning's: 91.5197 ft, Difference: 91.5197 ft
Feature ID: 1894376, Gage: 11313240, Depth by Geometry: 28.4338 ft, Depth by Manning's: 106.2614 ft, Difference: 77.8276 ft
Feature ID: 18696449, Gage: 02433151, Depth by Geometry: 22.9489 ft, Depth by Manning's: 83.4048 ft, Difference: 60.4559 ft
Feature ID: 22357431, Gage: 07380401, Depth by Geometry: 19.3477 ft, Depth by Manning's: 72.6633 ft, Difference: 53.3156 ft
Feature ID: 17037323, Gage: 09357700, Depth by Geometry: 23.1099 ft, Depth by Manning's: 75.9833 ft, Difference: 52.8735 ft


In [17]:
hist_values, bin_edges = np.histogram(diff, bins=5)

print("Histogram values:", hist_values)
print("Bin edges:", bin_edges)

Histogram values: [8657    1    1    0    1]
Bin edges: [   0.          256.34830692  512.69661384  769.04492075 1025.39322767
 1281.74153459]
